In [7]:
import os
from dotenv import load_dotenv

# Load credentials from .env file
load_dotenv()

# Verify keys are loaded
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is missing!"
assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY is missing!"
print("Environment keys loaded successfully.")

Environment keys loaded successfully.


In [6]:
from langchain_community.tools import TavilySearchResults

# Initialize the Tavily tool to search the web for recipes
search_tool = TavilySearchResults(max_results=3)

TypeError: ForwardRef._evaluate() missing 1 required keyword-only argument: 'recursive_guard'

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

# 1. Initialize the LLM (using gpt-4o-mini for speed and cost efficiency)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 2. Define the Agent's Prompt with a System message and place for memory
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional personal chef assistant. Your task is to help the user find delicious recipes using only the leftover ingredients they mention. Use the search tool to find recipes, summarize the instructions clearly, and answer any culinary or cooking follow-up questions."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="intermediate_steps"),
])

# 3. Create the agent with the tool
tools = [search_tool]
agent = create_tool_calling_agent(llm, tools, prompt)

# 4. Create the Executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

chat_history = []

def ask_chef(user_input: str):
    global chat_history
    
    # Run the agent with input and current chat history
    response = agent_executor.invoke({
        "input": user_input,
        "chat_history": chat_history
    })
    
    # Save the conversation to memory
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response["output"]))
    
    return response["output"]

In [ ]:
recipe_response = ask_chef("I have eggs, stale bread, and a tomato. What can I make?")
print("--- CHEF RESPONSE ---")
print(recipe_response)

# Follow-up question: Test the memory
follow_up = ask_chef("Can you give me a step-by-step guide on how to cook the first one?")
print("--- FOLLOW UP ANSWER ---")
print(follow_up)